In [1]:
!pip install xgboost optuna

In [1]:
import sys
!{sys.executable} -m pip install optuna xgboost

In [16]:
import sys
sys.path.append("../src")

In [17]:
from pipeline import preprocessor

In [21]:
import pandas as pd
import numpy as np
import optuna

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# load data
df = pd.read_parquet("../data/houses.parquet")

# feature engineering
df["Age"] = 2025 - df["YearBuilt"]
df["TotalBaths"] = df["FullBath"] + 0.5 * df["HalfBath"]
df["AreaPerRoom"] = df["GrLivArea"] / df["TotRmsAbvGrd"]

# ALL features (file wale)
features = [
    "LotArea","OverallQual","OverallCond","YearBuilt",
    "TotalBsmtSF","GrLivArea","FullBath","HalfBath",
    "GarageCars","GarageArea","TotRmsAbvGrd",
    "Age","TotalBaths","AreaPerRoom"
]

X = df[features]
y = df["SalePrice"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# optuna
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.2),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "random_state": 42
    }

    model = XGBRegressor(**params)
    model.fit(X_train, y_train)

    pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))

    return rmse

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

# final model
model = XGBRegressor(**study.best_params)
model.fit(X_train, y_train)

# save
import os, joblib
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/house_price_model.joblib")

print("✅ DONE")

[I 2026-04-17 15:12:43,537] A new study created in memory with name: no-name-81cf4f2a-3cd8-42ad-9330-03bde0b60d10
[I 2026-04-17 15:12:44,021] Trial 0 finished with value: 12373.403735431895 and parameters: {'n_estimators': 205, 'max_depth': 3, 'learning_rate': 0.1783161117174054, 'subsample': 0.9861452315491578, 'colsample_bytree': 0.839320886965331}. Best is trial 0 with value: 12373.403735431895.
[I 2026-04-17 15:12:44,877] Trial 1 finished with value: 12550.961078738153 and parameters: {'n_estimators': 354, 'max_depth': 4, 'learning_rate': 0.15990860442803528, 'subsample': 0.7629478965836695, 'colsample_bytree': 0.7213928906402146}. Best is trial 0 with value: 12373.403735431895.
[I 2026-04-17 15:12:45,517] Trial 2 finished with value: 12337.23923736587 and parameters: {'n_estimators': 323, 'max_depth': 3, 'learning_rate': 0.14200450917169527, 'subsample': 0.9027049016398424, 'colsample_bytree': 0.9158501800006498}. Best is trial 2 with value: 12337.23923736587.
[I 2026-04-17 15:12:

✅ DONE


In [24]:
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(model, "../models/house_price_model.joblib")

print("Model saved ✅")

Model saved ✅
